# Local Experts Pipeline — 100 Independent Pixel Models

Predicts the **t-1** state of every cell on a 10×10 GoL board by training one binary classifier per pixel.

**Phase 1** — train & save 100 models (`pixel_models/model_pixel_i.keras`).  
**Phase 2** — load all models, reconstruct full boards, evaluate comprehensively.

---
## 0 · Imports & Configuration

In [ ]:
import importlib
import functions
importlib.reload(functions)
from functions import *

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt

print(f'TensorFlow {tf.__version__}')

In [ ]:
# ─── Global hyper-parameters ──────────────────────────────────────────────────
SIZE          = 10
AMOUNT_BOARDS = 1000
GEN           = 4
N_PIXELS      = SIZE * SIZE   # 100

TEST_SIZE     = 0.10
VAL_SIZE      = 0.10
RANDOM_STATE  = 365

EPOCHS        = 50
BATCH_SIZE    = 256
ES_PATIENCE   = 5

MODEL_ROOT    = f'pixel_models/size{SIZE}_gen{GEN}'
os.makedirs(MODEL_ROOT, exist_ok=True)
print(f'Model root: {MODEL_ROOT}/')

---
## 1 · Load Raw Data Once

Features **X** are identical for every pixel — only the target column **y** changes.  
Load the raw DataFrame here so we avoid re-reading from disk 100 times.

In [ ]:
reverse_df = load_reverse_df(SIZE, AMOUNT_BOARDS, GEN)
print(f'Dataset shape: {reverse_df.shape}')

# ── Build the shared feature matrix (all gen-1 boards, flat) ──────────────────
amount_features = (GEN - 1) * SIZE * SIZE      # columns used as input
X_raw = reverse_df.iloc[:, :amount_features].astype(np.int8)

# ── Shared train / val / test INDEX split (same partitioning for every pixel) ─
idx = np.arange(len(X_raw))
idx_trainval, idx_test = train_test_split(idx, test_size=TEST_SIZE,  random_state=RANDOM_STATE)
idx_train,    idx_val  = train_test_split(idx_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE)

# 4-D numpy arrays for CNN  (n, SIZE, SIZE, gen-1)
X_train_4d = X_raw.iloc[idx_train].to_numpy().reshape(-1, SIZE, SIZE, GEN-1).astype(np.float32)
X_val_4d   = X_raw.iloc[idx_val  ].to_numpy().reshape(-1, SIZE, SIZE, GEN-1).astype(np.float32)
X_test_4d  = X_raw.iloc[idx_test ].to_numpy().reshape(-1, SIZE, SIZE, GEN-1).astype(np.float32)

print(f'Train: {X_train_4d.shape} | Val: {X_val_4d.shape} | Test: {X_test_4d.shape}')

# ── Full ground-truth boards at t-1  (n, 100) — used in Phase 2 ──────────────
target_cols = [f'Col_{amount_features + i}' for i in range(N_PIXELS)]
y_full = reverse_df[target_cols].astype(np.int8).to_numpy()  # (N, 100)

y_test_full = y_full[idx_test]      # (N_test, 100)  ← Phase 2 ground truth
print(f'y_test_full shape: {y_test_full.shape}')

---
## 2 · Architecture Registry

| Key          | Architecture              | Output       | Loss                         |
|--------------|---------------------------|--------------|------------------------------|
| `cnn`        | Lightweight CNN           | sigmoid (1)  | binary_crossentropy          |
| `rnn`        | Single LSTM               | sigmoid (1)  | binary_crossentropy          |
| `rcnn_sig`   | ConvLSTM2D                | sigmoid (1)  | binary_crossentropy          |
| `rcnn_soft`  | ConvLSTM2D                | softmax (2)  | sparse_categorical_crossentropy |

**Prediction rule**: sigmoid → `output >= 0.5`; softmax → `argmax(output)`

In [ ]:
TIMESTEPS = GEN - 1   # 3 board snapshots used as input

# ══════════════════════════════════════════════════════════════════════════════
# Reshape helpers  (N, SIZE, SIZE, TIMESTEPS) → model input
# ══════════════════════════════════════════════════════════════════════════════

def reshape_for_cnn(X):
    """(N, SIZE, SIZE, TIMESTEPS) — unchanged, channels = generations."""
    return X.astype(np.float32)

def reshape_for_rnn(X):
    """(N, TIMESTEPS, SIZE*SIZE) — each board flattened, sequence over gens."""
    n = X.shape[0]
    return X.transpose(0, 3, 1, 2).reshape(n, TIMESTEPS, SIZE * SIZE).astype(np.float32)

def reshape_for_rcnn(X):
    """(N, TIMESTEPS, SIZE, SIZE, 1) — spatial dims kept, channel dim added."""
    n = X.shape[0]
    return X.transpose(0, 3, 1, 2).reshape(n, TIMESTEPS, SIZE, SIZE, 1).astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# Model builders
# ══════════════════════════════════════════════════════════════════════════════

def build_cnn_pixel(input_shape):
    """Two Conv2D blocks → sigmoid.  ~7 k params."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_cnn')


def build_rnn_pixel(input_shape):
    """Single LSTM → sigmoid.  ~17 k params."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(32, activation='tanh'),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_rnn')


def _rcnn_base(input_shape, n_outputs, activation):
    """Shared ConvLSTM2D backbone — only the output head differs."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.ConvLSTM2D(
            filters=16, kernel_size=(3, 3),
            activation='relu', padding='same',
            return_sequences=True,
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ConvLSTM2D(
            filters=32, kernel_size=(3, 3),
            activation='relu', padding='same',
            return_sequences=False,
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(n_outputs, activation=activation),
    ], name=f'pixel_rcnn_{activation}')


def build_rcnn_sigmoid(input_shape):
    """ConvLSTM2D → sigmoid (1 output).  ~20 k params."""
    return _rcnn_base(input_shape, n_outputs=1, activation='sigmoid')


def build_rcnn_softmax(input_shape):
    """ConvLSTM2D → softmax (2 outputs: [P(dead), P(alive)]).  ~20 k params."""
    return _rcnn_base(input_shape, n_outputs=2, activation='softmax')


# ══════════════════════════════════════════════════════════════════════════════
# Registry — each entry carries build/reshape/loss/softmax flag
# ══════════════════════════════════════════════════════════════════════════════

ARCH_REGISTRY = {
    'cnn': {
        'build_fn':    build_cnn_pixel,
        'reshape_fn':  reshape_for_cnn,
        'input_shape': (SIZE, SIZE, TIMESTEPS),
        'loss':        'binary_crossentropy',
        'softmax':     False,
    },
    'rnn': {
        'build_fn':    build_rnn_pixel,
        'reshape_fn':  reshape_for_rnn,
        'input_shape': (TIMESTEPS, SIZE * SIZE),
        'loss':        'binary_crossentropy',
        'softmax':     False,
    },
    'rcnn_sig': {
        'build_fn':    build_rcnn_sigmoid,
        'reshape_fn':  reshape_for_rcnn,
        'input_shape': (TIMESTEPS, SIZE, SIZE, 1),
        'loss':        'binary_crossentropy',
        'softmax':     False,
    },
    'rcnn_soft': {
        'build_fn':    build_rcnn_softmax,
        'reshape_fn':  reshape_for_rcnn,
        'input_shape': (TIMESTEPS, SIZE, SIZE, 1),
        'loss':        'sparse_categorical_crossentropy',
        'softmax':     True,
    },
}

# Sanity-check: build each once and print param counts
print(f'{"Arch":<12}  {"Input shape":<30}  {"Params":>8}  {"Loss":<35}')
print('─' * 90)
for key, cfg in ARCH_REGISTRY.items():
    m = cfg['build_fn'](cfg['input_shape'])
    m.compile(optimizer='adam', loss=cfg['loss'])
    print(f'{key:<12}  {str(cfg["input_shape"]):<30}  {m.count_params():>8,}  {cfg["loss"]:<35}')
    del m
    tf.keras.backend.clear_session()

---
## PHASE 1 — Train All Architectures (100 models each)

In [ ]:
def get_callbacks(patience=ES_PATIENCE):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=patience,
            restore_best_weights=True, verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=max(1, patience // 2), min_lr=1e-5, verbose=0,
        ),
    ]


def compute_class_weights(y):
    y_flat = y.reshape(-1).astype(int)
    total  = len(y_flat)
    n0, n1 = int((y_flat == 0).sum()), int((y_flat == 1).sum())
    if n0 == 0 or n1 == 0:
        return None
    return {0: total / (2.0 * n0), 1: total / (2.0 * n1)}


def evaluate_full_boards(y_true, y_pred, architecture_name=''):
    """
    Full-board reconstruction metrics.
    y_true, y_pred : (N, 100) int  — ground-truth and predicted t-1 boards.
    Returns a metrics dict for the comparison table.
    """
    assert y_true.shape == y_pred.shape
    n_samples, _ = y_true.shape

    pixel_accs      = (y_true == y_pred).mean(axis=0)
    avg_pix_acc     = float(pixel_accs.mean())
    exact_matches   = (y_true == y_pred).all(axis=1)
    exact_board_acc = float(exact_matches.mean())
    n_exact         = int(exact_matches.sum())

    y_tf = y_true.flatten()
    y_pf = y_pred.flatten()
    precision = float(precision_score(y_tf, y_pf, zero_division=0))
    recall    = float(recall_score   (y_tf, y_pf, zero_division=0))
    f1        = float(f1_score       (y_tf, y_pf, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(y_tf, y_pf).ravel()

    sep = '─' * 54
    print(f'\n{sep}')
    print(f'  Architecture : {architecture_name}')
    print(f'  Test samples : {n_samples:,}')
    print(sep)
    print(f'  1. Avg Pixel-wise Accuracy    : {avg_pix_acc:.4f}  ({avg_pix_acc*100:.2f}%)')
    print(f'  2. Exact Board Match Accuracy : {exact_board_acc:.4f}  ({n_exact}/{n_samples})')
    print(f'  3. Alive-class  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}')
    print(f'     TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}')
    print(sep)

    return dict(
        architecture        = architecture_name,
        n_samples           = n_samples,
        avg_pixel_acc       = avg_pix_acc,
        exact_board_acc     = exact_board_acc,
        n_exact_matches     = n_exact,
        precision_alive     = precision,
        recall_alive        = recall,
        f1_alive            = f1,
        tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
        pixel_accs_per_pixel = pixel_accs.tolist(),
    )

In [ ]:
grand_log = {}

for arch_key, arch_cfg in ARCH_REGISTRY.items():

    arch_dir    = os.path.join(MODEL_ROOT, arch_key)
    os.makedirs(arch_dir, exist_ok=True)

    X_tr        = arch_cfg['reshape_fn'](X_train_4d)
    X_vl        = arch_cfg['reshape_fn'](X_val_4d)
    input_shape = arch_cfg['input_shape']
    is_softmax  = arch_cfg['softmax']

    print(f'\n{"═"*56}')
    print(f'  {arch_key.upper()} — input {input_shape}')
    print(f'  Loss: {arch_cfg["loss"]}  |  Output: {"softmax (2)" if is_softmax else "sigmoid (1)"}')
    print(f'{"═"*56}')

    arch_log = []
    t0_arch  = time.time()

    for pixel_idx in range(N_PIXELS):
        save_path = os.path.join(
            arch_dir,
            f'model_pixel{pixel_idx:03d}_size{SIZE}_gen{GEN}_{arch_key}.keras'
        )

        if os.path.exists(save_path):
            if pixel_idx % 10 == 0:
                print(f'  [pixel {pixel_idx:03d}] already exists, skipping ...')
            continue

        t0_pixel   = time.time()
        target_col = f'Col_{amount_features + pixel_idx}'
        y_all      = reverse_df[target_col].astype(np.float32).to_numpy()

        if is_softmax:
            y_train = y_all[idx_train].astype(np.int32)
            y_val   = y_all[idx_val  ].astype(np.int32)
        else:
            y_train = y_all[idx_train].reshape(-1, 1)
            y_val   = y_all[idx_val  ].reshape(-1, 1)

        model = arch_cfg['build_fn'](input_shape)
        model.compile(optimizer='adam', loss=arch_cfg['loss'], metrics=['accuracy'])

        history = model.fit(
            X_tr, y_train,
            validation_data=(X_vl, y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=get_callbacks(),
            class_weight=compute_class_weights(y_train),
            verbose=0,
        )
        model.save(save_path)

        best_val_acc = max(history.history['val_accuracy'])
        epochs_run   = len(history.history['loss'])
        elapsed      = time.time() - t0_pixel

        arch_log.append({
            'pixel_idx':     pixel_idx,
            'epochs_run':    epochs_run,
            'best_val_acc':  best_val_acc,
            'best_val_loss': min(history.history['val_loss']),
            'time_s':        elapsed,
        })

        if pixel_idx % 10 == 0 or pixel_idx == N_PIXELS - 1:
            print(f'  [{arch_key.upper()}] pixel {pixel_idx+1:3d}/100  '
                  f'val_acc={best_val_acc:.3f}  epochs={epochs_run:2d}  '
                  f'arch_total={time.time()-t0_arch:.0f}s')

        tf.keras.backend.clear_session()

    grand_log[arch_key] = arch_log
    print(f'\n  ✓ {arch_key.upper()} done — '
          f'{len(arch_log)} trained, {N_PIXELS-len(arch_log)} skipped')

print(f'\n{"═"*56}')
print('✓ All architectures complete.')

summary_rows = []
for arch_key, log in grand_log.items():
    if log:
        accs = [r['best_val_acc'] for r in log]
        summary_rows.append({
            'Arch': arch_key.upper(),
            'Output': 'softmax' if ARCH_REGISTRY[arch_key]['softmax'] else 'sigmoid',
            'Trained': len(log),
            'Mean val_acc': f'{np.mean(accs):.3f}',
            'Min':  f'{np.min(accs):.3f}',
            'Max':  f'{np.max(accs):.3f}',
        })
if summary_rows:
    display(pd.DataFrame(summary_rows).set_index('Arch'))

In [ ]:
# ─── Per-architecture training plots ─────────────────────────────────────────
archs_with_data = {k: v for k, v in grand_log.items() if v}

if archs_with_data:
    fig, axes = plt.subplots(2, len(archs_with_data),
                             figsize=(5 * len(archs_with_data), 7),
                             squeeze=False)

    for col, (arch_key, log) in enumerate(archs_with_data.items()):
        log_df = pd.DataFrame(log)

        # Val accuracy per pixel
        axes[0, col].bar(log_df['pixel_idx'], log_df['best_val_acc'], width=1.0)
        axes[0, col].axhline(log_df['best_val_acc'].mean(), color='red',
                             linestyle='--',
                             label=f"mean={log_df['best_val_acc'].mean():.3f}")
        axes[0, col].set_title(f'{arch_key.upper()} — Val Accuracy')
        axes[0, col].set_xlabel('Pixel index')
        axes[0, col].set_ylabel('Accuracy')
        axes[0, col].legend(fontsize=8)
        axes[0, col].set_ylim(0.5, 1.0)

        # Epochs per pixel
        axes[1, col].bar(log_df['pixel_idx'], log_df['epochs_run'],
                         width=1.0, color='orange')
        axes[1, col].set_title(f'{arch_key.upper()} — Early-Stop Epochs')
        axes[1, col].set_xlabel('Pixel index')
        axes[1, col].set_ylabel('Epochs')

    plt.tight_layout()
    plt.show()

---
## PHASE 2 — Evaluate All Architectures & Compare Full-Board Results

In [ ]:
all_arch_results = {}

for arch_key, arch_cfg in ARCH_REGISTRY.items():

    arch_dir   = os.path.join(MODEL_ROOT, arch_key)
    is_softmax = arch_cfg['softmax']

    # ── 1. Check all 100 files ────────────────────────────────────────────────
    missing = [
        i for i in range(N_PIXELS)
        if not os.path.exists(
            os.path.join(arch_dir,
                         f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{arch_key}.keras')
        )
    ]
    if missing:
        print(f'[{arch_key.upper():10s}] SKIP — {len(missing)} model(s) missing '
              f'(e.g. pixel {missing[0]:03d})')
        continue

    print(f'\n{"═"*56}')
    print(f'  {arch_key.upper()} — '
          f'{"softmax" if is_softmax else "sigmoid"} — loading 100 models ...')

    # ── 2. Load ───────────────────────────────────────────────────────────────
    t0 = time.time()
    models = [
        tf.keras.models.load_model(
            os.path.join(arch_dir,
                         f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{arch_key}.keras'),
            compile=False,
        )
        for i in range(N_PIXELS)
    ]
    print(f'  Loaded in {time.time()-t0:.1f}s  |  params/model: {models[0].count_params():,}')

    # ── 3. Reshape test set ───────────────────────────────────────────────────
    X_te = arch_cfg['reshape_fn'](X_test_4d)

    # ── 4. Predict — reconstruct full boards ──────────────────────────────────
    t0          = time.time()
    pred_boards = np.zeros((X_te.shape[0], N_PIXELS), dtype=np.int8)

    for i, model in enumerate(models):
        raw = model.predict(X_te, batch_size=512, verbose=0)
        if is_softmax:
            pred_boards[:, i] = np.argmax(raw, axis=1).astype(np.int8)
        else:
            pred_boards[:, i] = (raw.flatten() >= 0.5).astype(np.int8)
        if (i + 1) % 10 == 0 or i == N_PIXELS - 1:
            print(f'  Predicted pixel {i+1:3d}/100', end='\r')

    print(f'  Inference done in {time.time()-t0:.1f}s     ')
    del models
    tf.keras.backend.clear_session()

    # ── 5. Evaluate ───────────────────────────────────────────────────────────
    label = (f'{arch_key.upper()} '
             f'({"softmax" if is_softmax else "sigmoid"}, '
             f'size={SIZE}, gen={GEN})')
    all_arch_results[arch_key] = evaluate_full_boards(
        y_true=y_test_full,
        y_pred=pred_boards,
        architecture_name=label,
    )

print(f'\n✓ Evaluated {len(all_arch_results)} / {len(ARCH_REGISTRY)} architecture(s).')

---
## 3 · Summary of Results

In [ ]:
# ─── Comparison table ─────────────────────────────────────────────────────────
if not all_arch_results:
    print('No results yet — run Phase 1 + Phase 2 first.')
else:
    numeric_cols = ['Avg Pixel Acc', 'Exact Board Acc', 'Precision', 'Recall', 'F1']
    comparison_df = pd.DataFrame([
        {
            'Architecture':    r['architecture'],
            'Output':          'softmax' if ARCH_REGISTRY[k]['softmax'] else 'sigmoid',
            'Avg Pixel Acc':   r['avg_pixel_acc'],
            'Exact Board Acc': r['exact_board_acc'],
            'Exact Boards':    f"{r['n_exact_matches']}/{r['n_samples']}",
            'Precision':       r['precision_alive'],
            'Recall':          r['recall_alive'],
            'F1':              r['f1_alive'],
        }
        for k, r in all_arch_results.items()
    ]).set_index('Architecture')

    display(
        comparison_df.style
            .highlight_max(subset=numeric_cols, color='#c6efce')
            .highlight_min(subset=numeric_cols, color='#ffc7ce')
            .format({c: '{:.4f}' for c in numeric_cols})
            .set_caption(f'Local Experts — Full Board Comparison  (size={SIZE}, gen={GEN})')
    )

    print('\nBest architecture per metric:')
    for col in numeric_cols:
        best_idx = comparison_df[col].idxmax()
        print(f'  {col:<18}: {best_idx}  ({comparison_df.loc[best_idx, col]:.4f})')

    # ── Side-by-side per-pixel accuracy heatmaps ──────────────────────────────
    n = len(all_arch_results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
    for col, (arch_key, r) in enumerate(all_arch_results.items()):
        grid = np.array(r['pixel_accs_per_pixel']).reshape(SIZE, SIZE)
        im   = axes[0, col].imshow(grid, vmin=0.5, vmax=1.0, cmap='RdYlGn')
        axes[0, col].set_title(
            f'{arch_key.upper()}\n'
            f'{"softmax" if ARCH_REGISTRY[arch_key]["softmax"] else "sigmoid"} — '
            f'mean={r["avg_pixel_acc"]:.3f}', fontsize=9,
        )
        axes[0, col].set_xlabel('Col')
        axes[0, col].set_ylabel('Row')
        for row in range(SIZE):
            for c2 in range(SIZE):
                axes[0, col].text(c2, row, f'{grid[row,c2]:.2f}',
                                  ha='center', va='center', fontsize=5.5, color='black')
    fig.colorbar(im, ax=axes[0], label='Per-pixel accuracy', shrink=0.8)
    plt.suptitle(f'Per-pixel Accuracy — size={SIZE}, gen={GEN}', y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Bar chart ─────────────────────────────────────────────────────────────
    arch_labels = list(all_arch_results.keys())
    metrics_to_plot = {
        'Avg Pixel Acc':   [all_arch_results[k]['avg_pixel_acc']   for k in arch_labels],
        'Exact Board Acc': [all_arch_results[k]['exact_board_acc'] for k in arch_labels],
        'F1 (Alive)':      [all_arch_results[k]['f1_alive']        for k in arch_labels],
    }
    x, width = np.arange(len(arch_labels)), 0.25
    fig, ax = plt.subplots(figsize=(max(8, 2.5 * len(arch_labels)), 5))
    for (label, vals), offset, color in zip(
        metrics_to_plot.items(), [-width, 0, width], ['#4472C4', '#ED7D31', '#70AD47']
    ):
        bars = ax.bar(x + offset, vals, width, label=label, color=color, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels([k.upper() for k in arch_labels])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(f'CNN vs RNN vs RCNN-sig vs RCNN-soft  (size={SIZE}, gen={GEN})')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()